# Project 1 — Diabetes Diagnosis Prediction

## Professional Machine Learning Project

**Problem type:** Binary Classification  
**Target:** `Outcome`  
**Champion model:** Logistic Regression  
**Primary evaluation metric:** ROC-AUC  
**Model version:** 1.0.0

---

### Project Objective

Build and evaluate a reproducible machine learning classification pipeline for predicting the diabetes outcome represented by the `Outcome` variable.

This project emphasizes:

- data quality assessment
- preprocessing discipline
- leakage prevention
- model comparison
- explainability
- threshold analysis
- artifact integrity
- reproducibility
- responsible interpretation

> **Important:** This is an educational machine learning project. The model is not a clinical diagnostic system and must not be used as a substitute for professional medical evaluation.

## 1. Project Overview

This project follows a complete supervised machine learning workflow:

1. Load and preserve the source dataset.
2. Assess dataset quality.
3. Examine clinically suspicious zero values.
4. Separate features from the target.
5. Create a stratified train/test split.
6. Compare baseline and clinical-data preprocessing pipelines.
7. Evaluate both models on an untouched holdout set.
8. Select the champion model.
9. Explain the champion model.
10. Analyze classification thresholds.
11. Save the final model artifact.
12. Independently validate the saved artifact.

The final model artifact is stored separately from the notebook so that it can be validated independently.

In [ ]:
# Dataset loading — read-only

from pathlib import Path
import pandas as pd

DATASET_PATH = Path(
    "./Colab Notebooks/ML my Projects/"
    "Project_1_Diabetes_Diagnosis_Prediction/data/diabetes.csv"
)

df = pd.read_csv(DATASET_PATH)

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

display(df.head())

## 2. Problem Definition

### Analytical Question

Can patient-level attributes be used to estimate whether an observation belongs to the positive diabetes outcome class?

### Machine Learning Formulation

This is a **supervised binary classification** problem.

- **Features:** 8 numeric predictors
- **Target:** `Outcome`
- **Class 0:** negative outcome
- **Class 1:** positive outcome

### Evaluation Philosophy

ROC-AUC is the primary academic model-selection metric.
Recall, precision, F1, and accuracy are also reported because classification errors have different implications.

## 3. Dataset Description

The dataset contains **768 observations and 9 columns**.

There are:

- 8 predictor variables
- 1 binary target variable
- 500 observations in class 0
- 268 observations in class 1

All variables are numeric.

The original source dataset is preserved separately from the project working copy.

## 4. Data Quality Assessment

Initial structural validation found:

- **768 rows**
- **9 columns**
- **8 features**
- **0 pandas-detected missing values**
- **0 duplicate rows**

Several variables contain zero values that may be physiologically implausible.

### Suspicious Zero Counts

| Feature | Zero count | Percentage |
|---|---:|---:|
| Glucose | 5 | 0.65% |
| BloodPressure | 35 | 4.56% |
| SkinThickness | 227 | 29.56% |
| Insulin | 374 | 48.70% |
| BMI | 11 | 1.43% |

`Pregnancies = 0` is treated as a legitimate value.

A second pipeline therefore evaluates suspicious zeros as missing values followed by median imputation.

In [ ]:
# Exploratory analysis — read-only

import matplotlib.pyplot as plt

target_counts = df["Outcome"].value_counts().sort_index()

print("Target distribution:")
print(target_counts)

display(df.describe().T)

target_corr = (
    df.corr(numeric_only=True)["Outcome"]
    .drop("Outcome")
    .sort_values(ascending=False)
)

print("\nCorrelation with Outcome:")
display(target_corr.to_frame("correlation"))

plt.figure(figsize=(7, 4))
target_counts.plot(kind='bar')
plt.title('Target Distribution')
plt.xlabel('Outcome')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Train/Test Strategy

The dataset is divided using:

- **80% training**
- **20% testing**
- `random_state = 42`
- stratification on `Outcome`

Result:

| Split | Rows |
|---|---:|
| Training | 614 |
| Test | 154 |

The test set remains untouched during model fitting and preprocessing parameter estimation.

This prevents evaluation leakage.

## 6. Preprocessing

Two pipelines were evaluated.

### Baseline Pipeline

```text
Features
   ↓
StandardScaler
   ↓
LogisticRegression
```

### Clinical-data Preprocessing Pipeline

```text
Features
   ↓
Suspicious zero → NaN
   ↓
Median Imputation
   ↓
StandardScaler
   ↓
LogisticRegression
```

Both approaches use scikit-learn pipelines so preprocessing is fitted using training data only.

The target variable `Outcome` is excluded from the feature matrix.

In [ ]:
# Reproducible model comparison

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

TARGET = "Outcome"

SUSPICIOUS_ZERO_FEATURES = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
]

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

def replace_suspicious_zeros(frame):
    frame = frame.copy()
    frame[SUSPICIOUS_ZERO_FEATURES] = (
        frame[SUSPICIOUS_ZERO_FEATURES]
        .replace(0, np.nan)
    )
    return frame

baseline_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    )),
])

clinical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    )),
])

baseline_pipeline.fit(X_train, y_train)

X_train_clinical = replace_suspicious_zeros(X_train)
X_test_clinical = replace_suspicious_zeros(X_test)

clinical_pipeline.fit(
    X_train_clinical,
    y_train
)

baseline_pred = baseline_pipeline.predict(X_test)
baseline_prob = baseline_pipeline.predict_proba(X_test)[:, 1]

clinical_pred = clinical_pipeline.predict(X_test_clinical)
clinical_prob = clinical_pipeline.predict_proba(X_test_clinical)[:, 1]

def evaluate_model(y_true, pred, prob):
    return {
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, prob),
    }

comparison = pd.DataFrame({
    "Baseline Logistic Regression": evaluate_model(
        y_test, baseline_pred, baseline_prob
    ),
    "Clinical Preprocessing Logistic Regression": evaluate_model(
        y_test, clinical_pred, clinical_prob
    ),
}).T

display(comparison.round(4))

## 7. Model Evaluation

### Validated Holdout Results

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| **Baseline Logistic Regression** | **0.7143** | **0.6087** | **0.5185** | **0.5600** | **0.8230** |
| Clinical preprocessing Logistic Regression | 0.7078 | 0.6000 | 0.5000 | 0.5455 | 0.8130 |

The baseline pipeline outperformed the clinical preprocessing pipeline on every recorded evaluation metric.

### Baseline Confusion Matrix

- True Negatives: **82**
- False Positives: **18**
- False Negatives: **26**
- True Positives: **28**

The champion is therefore the **Baseline Logistic Regression** pipeline.

## 8. Champion Selection

### Champion Model

**Baseline Logistic Regression**

Pipeline:

```text
StandardScaler → LogisticRegression
```

Selection hierarchy:

1. ROC-AUC
2. Recall
3. F1
4. Accuracy

The baseline model achieved a ROC-AUC of **0.8230**, compared with **0.8130** for the clinical preprocessing alternative.

This selection is based on the evaluated holdout experiment and does not imply clinical superiority or deployment readiness.

## 9. Model Explainability

The champion model is logistic regression, allowing direct inspection of standardized coefficients and corresponding odds ratios.

| Feature | Coefficient | Odds Ratio |
|---|---:|---:|
| Glucose | 1.1442 | 3.1398 |
| BMI | 0.7139 | 2.0419 |
| Pregnancies | 0.3732 | 1.4523 |
| DiabetesPedigreeFunction | 0.2555 | 1.2911 |
| Age | 0.1842 | 1.2022 |
| SkinThickness | 0.0665 | 1.0688 |
| Insulin | -0.1273 | 0.8805 |
| BloodPressure | -0.1976 | 0.8207 |

Glucose has the strongest positive coefficient.

Because the features are standardized, the odds ratios correspond to approximately a one-standard-deviation increase in the associated feature.

These are statistical associations within the fitted model and dataset. They are not causal effects or medical diagnostic rules.

In [ ]:
# Champion coefficient inspection

champion = baseline_pipeline

feature_names = X.columns.tolist()

coefficients = (
    champion
    .named_steps["classifier"]
    .coef_[0]
)

explainability = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
})

explainability["Odds_Ratio"] = np.exp(
    explainability["Coefficient"]
)

explainability = explainability.sort_values(
    "Coefficient",
    ascending=False
)

display(explainability.round(4))

## 10. Threshold Analysis

The default classification threshold is 0.50.

The champion model was evaluated across thresholds from **0.20 to 0.80** in increments of 0.05.

Within this evaluated grid:

- Threshold **0.25** produced the highest recall: **0.8889**
- Threshold **0.25** produced the highest F1: **0.7007**
- Precision at 0.25: **0.5783**
- False Negatives at 0.25: **6**
- False Positives at 0.25: **35**

At the default threshold of 0.50:

- Precision: **0.6087**
- Recall: **0.5185**
- F1: **0.5600**
- False Negatives: **26**
- False Positives: **18**

### Important Limitation

No clinical operating threshold is selected.

Threshold selection requires intended-use analysis, error-cost analysis, calibration, and appropriate clinical validation.

In [ ]:
# Threshold analysis

from sklearn.metrics import confusion_matrix

threshold_rows = []

for threshold in np.arange(0.20, 0.801, 0.05):

    threshold = round(float(threshold), 2)

    pred = (baseline_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        pred
    ).ravel()

    threshold_rows.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
    })

threshold_results = pd.DataFrame(threshold_rows)

display(threshold_results.round(4))

## 11. Final Model Artifact

The final champion model was serialized as:

`artifacts/diabetes_baseline_logistic_regression.joblib`

### Model Version

**1.0.0**

### Model SHA256

`5db1b088d1bfb18c74c9e7622e013c4258759f9f6f9e34a6a4d4398f8febba9c`

### Dataset SHA256

`698c203a14aa31941d2251175330c9199f3ccdb31597abbba2a3e35416257a72`

The hashes provide integrity checks for future reproducibility validation.

In [ ]:
# Independent artifact validation

import joblib

ARTIFACT_PATH = Path(
    "./Colab Notebooks/ML my Projects/"
    "Project_1_Diabetes_Diagnosis_Prediction/artifacts/"
    "diabetes_baseline_logistic_regression.joblib"
)

loaded_model = joblib.load(ARTIFACT_PATH)

loaded_pred = loaded_model.predict(X_test)
loaded_prob = loaded_model.predict_proba(X_test)[:, 1]

validation_metrics = {
    "Accuracy": accuracy_score(y_test, loaded_pred),
    "Precision": precision_score(y_test, loaded_pred, zero_division=0),
    "Recall": recall_score(y_test, loaded_pred, zero_division=0),
    "F1": f1_score(y_test, loaded_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, loaded_prob),
}

print("Artifact exists:", ARTIFACT_PATH.exists())

print("\nReproduced metrics:")

for metric, value in validation_metrics.items():
    print(f"{metric:10s}: {value:.6f}")

## 12. Reproducibility Validation

Independent validation confirmed:

- artifact exists
- model hash matches
- dataset hash matches
- pipeline structure matches
- Logistic Regression classifier is present
- eight expected features are present
- 154 test predictions are reproduced
- 154 probability predictions are reproduced
- predictions are valid
- probabilities are valid
- recorded holdout metrics are reproduced

### Reproduced Metrics

| Metric | Value |
|---|---:|
| Accuracy | 0.714286 |
| Precision | 0.608696 |
| Recall | 0.518519 |
| F1 | 0.560000 |
| ROC-AUC | 0.822963 |

This demonstrates that the serialized model can be independently loaded and produces the expected results.

## 13. Limitations

### Dataset Limitations

The dataset contains only 768 observations and includes zero-coded values that may represent missing measurements.

### Statistical Limitations

The evaluation uses one held-out test set containing 154 observations. Performance therefore contains sampling uncertainty.

### Modeling Limitations

This project evaluates a focused set of classical modeling choices. External validation, extensive hyperparameter optimization, calibration analysis, and ensemble methods are outside the current scope.

### Clinical Limitations

The model has not undergone prospective clinical validation, external clinical validation, clinical calibration validation, regulatory review, clinical workflow validation, or comprehensive subgroup evaluation.

Therefore, this model is an educational machine learning artifact and not a clinical diagnostic instrument.

## 14. Responsible AI Considerations

Model performance should not be confused with real-world clinical usefulness.

Important considerations for a future production system include:

- false-negative consequences
- false-positive consequences
- population shift
- missing-data behavior
- calibration
- subgroup performance
- fairness
- interpretability
- monitoring
- human oversight

The threshold analysis illustrates why an operating threshold should not be selected solely by optimizing one metric on one holdout dataset.

# 15. Final Conclusion

The project successfully developed and validated a reproducible binary classification pipeline for diabetes outcome prediction.

The **Baseline Logistic Regression** pipeline was selected as the champion because it outperformed the alternative clinical-data preprocessing pipeline across the evaluated holdout metrics.

### Final Validated ROC-AUC

> **0.8230**

The final model was serialized, hashed, and independently validated.

The project demonstrates the complete machine learning lifecycle:

**data → quality assessment → preprocessing → modeling → evaluation → explainability → threshold analysis → artifact creation → reproducibility**

### Final Status

**PROJECT STATUS: PASS**

**MODEL STATUS: VALIDATED**

**ARTIFACT STATUS: VERIFIED**

**CLINICAL DEPLOYMENT STATUS: NOT CLAIMED**

## Appendix — Project Evidence

The following evidence files were generated during Steps 1–10:

- `project_structure_evidence.json`
- `dataset_quality_evidence.json`
- `clinical_eda_evidence.json`
- `preprocessing_split_evidence.json`
- `model_pipeline_evidence.json`
- `model_evaluation_evidence.json`
- `model_explainability_evidence.json`
- `threshold_analysis_evidence.json`
- `final_model_artifact_evidence.json`
- `reproducibility_artifact_validation_evidence.json`
- `notebook_packaging_evidence.json`

These evidence files provide traceability between the notebook and the validated project artifacts.